### Visualize Derivatives

#### 1. First Line: create a tensor that tracks gradients

```
x = torch.arange(-8.0,8.0,0.1, requires_grad=True)
```

This creates a tensor:

$$
x=[-8.0,-7.9,-7.8, \ldots, 7.9]
$$


```
requires_grad=True
```

This tells PyTorch:
Track all operations involving this tensor so we can compute derivatives later.
Internally PyTorch builds a computation graph.

#### 2. Apply ReLU

```
y=torch.relu(x)
```

Mathematically:

$$
y_i=\max \left(0, x_i\right)
$$


Example:

| $x$ | y |
| :--- | :--- |
| -2 | 0 |
| -1 | 0 |
| 0.5 | 0.5 |
| 2 | 2 |

But because `x` requires gradients, PyTorch records:
```
x -> relu -> y
```


So it knows how to compute:

$$
\frac{d y}{dx}
$$

#### 3. Plot the function

```
d2l.plot(x.detach(), y.detach(), 'x', 'relu(x)')
```

Important part:
```
.detach()
```

This removes tensors from the computation graph.

Why?

Because plotting libraries (matplotlib) expect normal arrays, not tensors tracking gradients.
So we plot:
```
x values
y values
```
This gives the ReLU curve.

#### 4. Compute gradients

Now the interesting part:
```
y.backward(torch.ones_like(x), retain_graph=True)
```
Let's unpack it.

##### What backward normally does

Normally you see:
```
y.backward()
```

But that only works if $y$ is a scalar.
Example:

$$
y=x^2
$$


Then PyTorch computes:

$$
\frac{d y}{d x}
$$

##### But here y is a vector

Here:
```
y = relu(x)
```
So:
```
x : shape $(160$,
y : shape $(160$,
```
So PyTorch asks:
derivative of WHICH scalar?

Instead we compute a vector-Jacobian product.

##### What `torch.ones_like(x)` means

We are telling PyTorch:
Compute

$$
\sum_i y_i
$$


Because:
```
y.backward(ones)
```
is equivalent to:

$$
\frac{d}{d x} \sum_i y_i
$$


Since:

$$
y_i=\operatorname{relu}\left(x_i\right)
$$

we get:

$$
\frac{d}{d x_i} \operatorname{relu}\left(x_i\right)
$$

which is exactly the ReLU derivative.

#### 5. Where the gradient is stored

After backward:
```
x.grad
```
contains:

$$
\frac{d y}{d x}
$$

elementwise.
So:

| x | grad |
| :--- | :--- |
| -2 | 0 |
| -1 | 0 |
| 0.5 | 1 |
| 2 | 1 |

#### 6. Why `retain_graph = True`

```
y.backward(..., retain_graph=True)
```

Normally PyTorch destroys the computation graph after backward to save memory.
But if you want to:
- call backward again
- inspect the graph

you must keep it.
So D2L adds this for safety.